# Getting started with Claude Opus 4.7 (Nutmeg) on Amazon Bedrock

**Anthropic's most capable Opus model yet — advancing agentic coding, knowledge work, and long-running tasks**

This notebook demonstrates how to use Claude Opus 4.7 on Amazon Bedrock. Opus 4.7 is a direct upgrade from Opus 4.6, with improvements across agentic coding, professional knowledge work, long-horizon tasks, cross-session memory, and high-resolution vision.

---

## What you'll learn

- **Basic usage**: Get started with Claude Opus 4.7 using the Converse, InvokeModel, and Messages APIs
- **Adaptive thinking**: Leverage Claude's intelligent extended thinking mode with configurable effort levels
- **Context compaction**: Enable infinite context for long-running conversations (beta)

---

## When to use Claude Opus 4.7

Claude Opus 4.7 is ideal for:

- **Agentic coding** — stronger performance on long-horizon autonomy, systems engineering, and complex code reasoning
- **Knowledge work** — document creation, financial analysis, and multi-step research workflows
- **Long-running tasks** — multi-step tool use, error recovery, and reasoning across the full 1M-token context
- **Autonomous agents** — better reasoning through ambiguity and stronger memory for longer unsupervised runs
- **Vision tasks** — high-resolution image support for screenshots, diagrams, and design files

---

## Key capabilities

| Capability | Details |
|------------|--------|
| **Context window** | Up to 1M tokens |
| **Max output** | Up to 128K tokens |
| **Adaptive thinking** | Extended reasoning that activates when needed, with configurable effort levels |
| **Context compaction** | Automatic summarization for infinite context (beta, InvokeModel only) |
| **Memory** | Cross-session memory — knows when to write context down and when to recall it |
| **Vision** | High-resolution image input for finer detail in screenshots, diagrams, and design files |

---

## API options on Amazon Bedrock

Claude Opus 4.7 is available through **Amazon Bedrock Mantle**, a new endpoint that exposes the Anthropic Messages API natively. You can invoke the model through three API paths:

| API | Endpoint | When to use |
|-----|----------|-------------|
| **Messages API** | `bedrock-mantle.{region}.api.aws/anthropic/v1/messages` | Direct access to Anthropic's native API format via Bedrock Mantle. Full feature support. |
| **InvokeModel API** | `bedrock-runtime.{region}.amazonaws.com` | Bedrock's native API with Anthropic request body format. Supports all features including compaction. |
| **Converse API** | `bedrock-runtime.{region}.amazonaws.com` | Bedrock's unified conversational API. Simplest integration, works across Bedrock models. |

All three APIs are fully supported. The Messages API via Bedrock Mantle uses the same request/response shape as Anthropic's first-party API, making it easy to port existing code. The InvokeModel and Converse APIs go through the standard `bedrock-runtime` endpoint and are mapped internally by Bedrock Mantle.

---

## 1. Setup and configuration

In [ ]:
# Install required packages (uncomment if needed)
# !pip install boto3 --upgrade
# !pip install anthropic[bedrock] --upgrade  # For Messages API via Bedrock Mantle

In [ ]:
import boto3
import json
from botocore.config import Config

# Configuration
REGION = "us-east-1"
MODEL_ID = "us.anthropic.claude-opus-4-7-v1"

# Config with longer timeout for thinking/compaction
FAST_CONFIG = Config(read_timeout=1200, retries={'max_attempts': 1})

# Initialize session
session = boto3.Session()

# Initialize the Bedrock Runtime client
bedrock_runtime = session.client(
    service_name='bedrock-runtime',
    region_name=REGION,
    config=FAST_CONFIG
)

print(f"✓ Region: {REGION}")
print(f"✓ Model: {MODEL_ID}")
print(f"✓ Client initialized successfully")

---

## 2. Basic usage with Converse API

The Converse API provides a unified interface for conversational AI across Amazon Bedrock models. This is the simplest way to get started.

In [ ]:
# Basic request using the Converse API
response = bedrock_runtime.converse(
    modelId=MODEL_ID,
    messages=[
        {
            "role": "user",
            "content": [{"text": "What are three key considerations when designing a distributed system for high availability?"}]
        }
    ],
    inferenceConfig={
        "maxTokens": 2048,
        "temperature": 1
    }
)

print(response["output"]["message"]["content"][0]["text"])

---

## 3. Basic usage with InvokeModel API

The InvokeModel API provides direct access to Claude's native request format and supports the full set of Opus 4.7 features.

In [ ]:
# Basic request using the InvokeModel API
request_body = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 2048,
    "temperature": 1,
    "messages": [
        {
            "role": "user",
            "content": [{"type": "text", "text": "Explain the CAP theorem and its implications for database design."}]
        }
    ]
}

response = bedrock_runtime.invoke_model(
    modelId=MODEL_ID,
    body=json.dumps(request_body)
)

response_body = json.loads(response["body"].read())
print(response_body["content"][0]["text"])

---

## 4. Basic usage with Messages API (Bedrock Mantle)

The Messages API is available through the Bedrock Mantle endpoint. It uses the same request/response format as Anthropic's first-party API, authenticated with AWS SigV4. This is the most direct way to access Claude's full capabilities.

You can use the `anthropic[bedrock]` SDK package for a streamlined experience:

In [ ]:
from anthropic import AnthropicBedrockMantle

# Initialize the Bedrock Mantle client (uses SigV4 auth automatically)
mantle_client = AnthropicBedrockMantle(aws_region=REGION)

# Create a message using the Messages API
message = mantle_client.messages.create(
    model="anthropic.claude-opus-4-7-v1",
    max_tokens=2048,
    messages=[
        {"role": "user", "content": "What are the key differences between eventual and strong consistency in distributed systems?"}
    ]
)

print(message.content[0].text)

---

## 5. Adaptive thinking

Adaptive thinking gives Claude the freedom to think if and when it determines reasoning is required. Instead of fixed token budgets, you guide Claude's reasoning depth with an `effort` parameter.

### Effort levels

| Effort | Description |
|--------|-------------|
| **max** | Maximum reasoning depth. Reserve for the most complex analytical workloads. |
| **high** | Default. Deep reasoning on complex tasks. Start here. |
| **medium** | Balanced. May skip thinking for straightforward queries. |
| **low** | Minimal thinking, prioritizes speed and cost. |

### Adaptive thinking with Converse API

With the Converse API, adaptive thinking is configured using `additionalModelRequestFields`:

In [ ]:
# Adaptive thinking with Converse API
response = bedrock_runtime.converse(
    modelId=MODEL_ID,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "text": "Design a fault-tolerant distributed consensus algorithm that handles Byzantine failures in an asynchronous network with up to 1/3 faulty nodes."
                }
            ]
        }
    ],
    inferenceConfig={
        "maxTokens": 50000,
        "temperature": 1
    },
    additionalModelRequestFields={
        "thinking": {
            "type": "adaptive"
        }
    }
)

# Check if thinking was triggered
output_message = response["output"]["message"]
has_thinking = any(block.get("type") == "thinking" for block in output_message["content"])
print(f"Claude decided to think: {has_thinking}\n")

# Display content blocks
for block in output_message["content"]:
    if block.get("type") == "thinking":
        print("🧠 Thinking (first 300 chars):")
        print(block["thinking"][:300] + "...\n")
    elif block.get("text"):
        print("💬 Response:")
        text = block["text"]
        print(text[:800] + "..." if len(text) > 800 else text)

print(f"\n📊 Usage: Input tokens={response['usage']['inputTokens']}, Output tokens={response['usage']['outputTokens']}")

### Adaptive thinking with InvokeModel API

In [ ]:
# Adaptive thinking with InvokeModel API and effort control
request_body = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 16000,
    "temperature": 1,
    "thinking": {
        "type": "adaptive",
        "effort": "high"
    },
    "messages": [
        {
            "role": "user",
            "content": "I have a distributed system where Service A writes to DynamoDB and publishes to SNS, and Service B consumes from SQS and writes to Aurora PostgreSQL. During peak load, I'm seeing occasional data inconsistencies. Walk me through the possible race conditions and give me a design that guarantees eventual consistency without distributed transactions."
        }
    ]
}

response = bedrock_runtime.invoke_model(
    modelId=MODEL_ID,
    body=json.dumps(request_body)
)

response_body = json.loads(response["body"].read())

# IMPORTANT: Handle both thinking and text blocks
# Claude may or may not include thinking depending on the query
for block in response_body["content"]:
    if block["type"] == "thinking":
        print(f"🧠 Thinking (first 300 chars): {block['thinking'][:300]}...")
    elif block["type"] == "text":
        print(f"💬 Response: {block['text'][:800]}..." if len(block['text']) > 800 else f"💬 Response: {block['text']}")

### Comparing effort levels

Different effort levels produce different reasoning depths. Higher effort may trigger more thinking on complex prompts:

In [ ]:
# Compare effort levels on the same prompt
def test_effort_level(effort_level: str, prompt: str):
    """Test a specific effort level and return thinking stats"""
    request_body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 12000,
        "temperature": 1,
        "thinking": {"type": "adaptive"},
        "output_config": {"effort": effort_level},
        "messages": [{"role": "user", "content": prompt}]
    }
    response = bedrock_runtime.invoke_model(modelId=MODEL_ID, body=json.dumps(request_body))
    response_body = json.loads(response["body"].read())

    has_thinking = False
    thinking_length = 0
    response_length = 0
    for block in response_body["content"]:
        if block["type"] == "thinking":
            has_thinking = True
            thinking_length = len(block["thinking"])
        elif block["type"] == "text":
            response_length = len(block["text"])
    return {"effort": effort_level, "thought": has_thinking, "thinking_chars": thinking_length, "response_chars": response_length}

prompt = "What are the key considerations when implementing a caching strategy for a high-traffic API?"
print("Testing different effort levels on the same prompt...\n")

for effort in ["low", "medium", "high", "max"]:
    result = test_effort_level(effort, prompt)
    print(f"Effort: {result['effort']:6} | Thought: {str(result['thought']):5} | "
          f"Thinking: {result['thinking_chars']:5} chars | Response: {result['response_chars']:5} chars")

print("\n✓ Higher effort levels may trigger more thinking on complex prompts")

---

## 6. Context compaction (beta)

Context compaction enables "infinite context" for long-running conversations and agentic tasks by automatically summarizing older context when approaching the context window limit.

**How it works:**
1. Enable compaction with `context_management.edits` containing a `compact_20260112` edit
2. When input tokens exceed a trigger threshold, Claude generates a summary
3. The API returns a compaction content block
4. Pass the compaction block back as part of the messages array on subsequent requests

**Important:**
- Context compaction is only available with the InvokeModel API (not Converse API)
- Requires `anthropic_beta: ["compact-2026-01-12"]` header
- Default trigger is ~500K tokens; customize with `trigger` parameter

In [ ]:
# Basic compaction example
request_body = {
    "anthropic_version": "bedrock-2023-05-31",
    "anthropic_beta": ["compact-2026-01-12"],
    "max_tokens": 4096,
    "temperature": 1,
    "messages": [
        {
            "role": "user",
            "content": "Let's start a long-running conversation about building a complex e-commerce platform."
        }
    ],
    "context_management": {
        "edits": [
            {"type": "compact_20260112"}
        ]
    }
}

response = bedrock_runtime.invoke_model(
    modelId=MODEL_ID,
    body=json.dumps(request_body)
)

response_body = json.loads(response["body"].read())

# Check for compaction block
has_compaction = any(block.get("type") == "compaction" for block in response_body["content"])
print(f"Compaction triggered: {has_compaction}")

for block in response_body["content"]:
    if block["type"] == "text":
        print(f"\n💬 Response:\n{block['text'][:500]}..." if len(block['text']) > 500 else f"\n💬 Response:\n{block['text']}")
    elif block.get("type") == "compaction":
        print("\n📦 Compaction block received — pass this back on subsequent requests")

print(f"\n📊 Usage: {response_body['usage']}")

In [ ]:
# Multi-turn conversation with compaction and custom trigger
messages = []

def chat(user_message: str):
    """Send a message and handle compaction automatically"""
    messages.append({"role": "user", "content": user_message})

    request_body = {
        "anthropic_version": "bedrock-2023-05-31",
        "anthropic_beta": ["compact-2026-01-12"],
        "max_tokens": 4096,
        "messages": messages,
        "context_management": {
            "edits": [{
                "type": "compact_20260112",
                "trigger": {
                    "type": "input_tokens",
                    "value": 100000  # Trigger at 100K tokens
                }
            }]
        }
    }

    response = bedrock_runtime.invoke_model(
        modelId=MODEL_ID,
        body=json.dumps(request_body)
    )

    response_body = json.loads(response["body"].read())
    messages.append({"role": "assistant", "content": response_body["content"]})

    for block in response_body["content"]:
        if block["type"] == "text":
            return block["text"]
    return ""

# Have a multi-turn conversation
print("Turn 1:")
print(chat("Design the database schema for an e-commerce platform")[:500] + "...")
print("\n" + "="*50 + "\n")
print("Turn 2:")
print(chat("Now add the authentication and authorization system")[:500] + "...")

---

## Next steps

- Explore [Claude Opus 4.7 documentation](https://docs.aws.amazon.com/bedrock/latest/userguide/) on Amazon Bedrock
- Try different effort levels on your specific workloads to find the right balance of quality vs. speed
- For long-running agents, enable compaction to avoid context window limits
- Check out the [Anthropic SDK for Bedrock Mantle](https://docs.anthropic.com/en/api/claude-on-amazon-bedrock) for Messages API access